# Sigiriya SinLlama inference

Select **Runtime > Change runtime type > T4 GPU**. Run the cells in order.
Upload the rebuilt `sigiriya_sinllama_focused_runtime_v3.zip`. It contains the saved
evidence packet. To use a fresh export, also upload `sigiriya_sinllama_evidence.json`.

The default selection reproduces the two verified v3 control configurations.
Choose profiles by evidence question ID in the selection cell. For the
saved `multi_topic` question, explicitly choose A, B, or C. All three profiles
used the extractive fallback for both Q03 topics in the recorded T4 run.


In [ ]:
%pip install -q transformers==4.57.1 accelerate==1.10.1 bitsandbytes==0.48.2

In [ ]:
from io import BytesIO
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile

import torch
from google.colab import files

assert torch.cuda.is_available(), "Select a GPU runtime before generating answers."
print(torch.cuda.get_device_name())
bundle_name = "sigiriya_sinllama_focused_runtime_v3.zip"
uploaded = files.upload()
assert bundle_name in uploaded, f"Upload {bundle_name}"
workspace = Path(mkdtemp(prefix="sigiriya_focused_v3_", dir="/content"))
with ZipFile(BytesIO(uploaded[bundle_name])) as archive:
    archive.extractall(workspace)

In [ ]:
import json
import sys

sys.path.insert(0, str(workspace))
from scripts.sinllama_diagnostic_v1 import EVIDENCE
from scripts.sinllama_focused_comparison_v3 import PROFILES

evidence_name = EVIDENCE.name
evidence_bytes = uploaded.get(evidence_name)
if evidence_bytes is None:
    evidence_bytes = EVIDENCE.read_bytes()
payload = json.loads(evidence_bytes.decode("utf-8-sig"))

for row in payload["questions"]:
    print(row["id"], row["question"], row["status"])
print(PROFILES)

# Edit this mapping for the question IDs shown above.
# A, B, C use the packet's ordered, decomposed topic evidence.
# R-Q01 and R-Q05 use the whole evidence row.
selection = {"factual": "R-Q01", "mechanism": "R-Q05"}
# selection["multi_topic"] = "A"  # Choose A, B, or C explicitly.


In [ ]:
from hashlib import sha256

from scripts.sinllama_focused_comparison_v3 import (
    REVISION, FocusedComparisonGenerator, run_selected,
)
from scripts.sinllama_handoff import write_json
from scripts.sinllama_legacy_rag_comparison_v1 import LEGACY_V4
from scripts.sinllama_reporting import print_report
from src.config import SINLLAMA_MODEL

model = FocusedComparisonGenerator(
    config=LEGACY_V4, revision=REVISION, local_files_only=False
)
records = list(run_selected(payload, selection, model.generate_unit))
questions = {row["id"]: row["question"] for row in payload["questions"]}
print_report(records, questions=questions)

report_path = workspace / "sigiriya_sinllama_answers.json"
write_json(report_path, {
    "checkpoint": SINLLAMA_MODEL,
    "checkpoint_revision": REVISION,
    "evidence_sha256": sha256(evidence_bytes).hexdigest(),
    "profiles": selection,
    "final_answer_policy": "strict",
    "results": records,
})


In [ ]:
files.download(str(report_path))
